# Lesson 03: Chunking, Overlap & Vector Search (L2 Distance)
> **Goal:** Understand the 4 foundational steps of every RAG pipeline:
> 1. **Chunking:** Slicing long documents into bite-sized segments.
> 2. **Overlap:** Preventing sentences from getting cut in half at chunk borders.
> 3. **Vectors:** Converting text chunks into coordinates in space.
> 4. **Euclidean (L2) Distance:** Finding the closest chunk to a user's question.

---

```mermaid
flowchart TD
    A["Raw Document (data.txt)"] --> B["Chunking with Overlap<br/>(size=40, overlap=10)"]
    B --> C["27 Document Chunks"]
    C --> D["Vector Embedding (16D Space)"]
    
    Q["User Query (question.txt)"] --> E["Vector Embedding (16D Space)"]
    
    D & E --> F["Euclidean (L2) Distance<br/>sqrt(sum((q - c)^2))"]
    F --> G["Lowest Distance = Best Match!"]
```


### The Intuition: Euclidean (L2) Distance
Euclidean distance is simply the familiar **Pythagorean theorem** applied to vectors:

$$d(\mathbf{q}, \mathbf{c}) = \sqrt{\sum_{i=1}^D (q_i - c_i)^2}$$

- **$d = 0.0$:** The vectors are identical (exact match).
- **Smaller distance = Closer match:** Unlike similarity scores where higher is better, in distance metrics, **smaller is better**.
- In Python with NumPy, this is computed across all vectors in one operation: `np.linalg.norm(chunk_vectors - query_vec, axis=1)`.


In [2]:
# Step 1 & 2: Load document and slice into chunks with overlap
from pathlib import Path

for p in [Path("data.txt"), Path("src/notebooks/data.txt"), Path("../data.txt")]:
    if p.exists():
        data_file = p
        break
else:
    raise FileNotFoundError("Could not find data.txt")

with open(data_file, "r", encoding="utf-8") as f:
    text = f.read()

def chunk_text(text: str, chunk_size: int = 40, overlap: int = 10) -> list[str]:
    """Slices text into chunks of `chunk_size` words with `overlap` shared words."""
    words = text.split()
    chunks = []
    step = chunk_size - overlap  # Advances forward by 30 words each step
    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        chunks.append(" ".join(chunk_words))
    return chunks

chunks = chunk_text(text, chunk_size=40, overlap=10)
total_words = len(text.split())
print(f"Total document words: {total_words} -> Created {len(chunks)} overlapping chunks.")


Total document words: 781 -> Created 27 overlapping chunks.


In [3]:
# Step 3: Inspect the seam between Chunk #1 and Chunk #2 to see the overlap
chunk1_end = " ".join(chunks[0].split()[-10:])
chunk2_start = " ".join(chunks[1].split()[:10])

print("--- End of Chunk #1 (last 10 words) ---")
print(f"... {chunk1_end}")

print("\n--- Start of Chunk #2 (first 10 words) ---")
print(f"{chunk2_start} ...")

print("\nKey Insight: The 10 shared words guarantee that sentences on the seam are never lost!")


--- End of Chunk #1 (last 10 words) ---
... during an internal executive summit, the leadership team initiated a

--- Start of Chunk #2 (first 10 words) ---
during an internal executive summit, the leadership team initiated a ...

Key Insight: The 10 shared words guarantee that sentences on the seam are never lost!


In [4]:
# Step 4: Convert chunks into 16-dimensional dense vectors locally
# (We use local TF-IDF + SVD so you can see vector coordinates without an API key)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import numpy as np

tfidf = TfidfVectorizer(stop_words="english")
svd = TruncatedSVD(n_components=16, random_state=42)

# Create 16-dimensional coordinate vectors for every chunk
chunk_vectors = svd.fit_transform(tfidf.fit_transform(chunks))

print(f"Chunk vectors shape: {chunk_vectors.shape} ({chunk_vectors.shape[0]} chunks x {chunk_vectors.shape[1]} dimensions)")
print(f"Sample vector for Chunk #1 (first 4 dims): {np.round(chunk_vectors[0][:4], 3)}")


Chunk vectors shape: (27, 16) (27 chunks x 16 dimensions)
Sample vector for Chunk #1 (first 4 dims): [0.245 0.241 0.251 0.08 ]


In [5]:
# Step 5: Search using Euclidean (L2) Distance
# Intuition: Smallest distance = Closest match!

def search_l2(query: str, top_k: int = 2) -> list[dict]:
    # 1. Project the query into the exact same 16-dimensional space
    query_vec = svd.transform(tfidf.transform([query]))
    
    # 2. Calculate Euclidean distance from query to every chunk
    distances = np.linalg.norm(chunk_vectors - query_vec, axis=1)
    
    # 3. Sort ascending: smallest distance comes first
    top_indices = np.argsort(distances)[:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "chunk_id": int(idx + 1),
            "distance": float(distances[idx]),
            "text": chunks[idx]
        })
    return results

print("L2 Vector Search Function Ready!")


L2 Vector Search Function Ready!


In [6]:
# Step 6: Test vector search on questions from question.txt
from pathlib import Path

for p in [Path("question.txt"), Path("src/notebooks/question.txt"), Path("../question.txt")]:
    if p.exists():
        question_file = p
        break
else:
    raise FileNotFoundError("Could not find question.txt")

with open(question_file, "r", encoding="utf-8") as f:
    questions = [line.strip() for line in f if line.strip()]

test_queries = [
    questions[0],  # SKU / product launch question
    questions[2]   # Fiscal year 2023 revenue and EBITDA question
]

for q in test_queries:
    print(f"QUERY: \"{q}\"")
    matches = search_l2(q, top_k=2)
    for rank, match in enumerate(matches, 1):
        print(f"  Rank {rank} (L2 Distance: {match['distance']:.4f}) -> Chunk #{match['chunk_id']}:")
        print(f"     \"{match['text'][:85]}...\"")
    print()


QUERY: "What is the catalog part number for the Titan-X9 processor and how much revenue did it generate in its first month?"
  Rank 1 (L2 Distance: 0.6036) -> Chunk #20:
     "defect rates. On April 15, 2024, Aura officially commenced commercial availability of..."
  Rank 2 (L2 Distance: 0.6972) -> Chunk #19:
     "2024, compared to 890 workers a year earlier. First-quarter 2024 financial filings re..."

QUERY: "What was Aura's total consolidated revenue and adjusted EBITDA for the full fiscal year 2023?"
  Rank 1 (L2 Distance: 0.4795) -> Chunk #14:
     "in quarterly gross top-line volume. For the complete fiscal year 2023, Aura reported ..."
  Rank 2 (L2 Distance: 0.6713) -> Chunk #13:
     "autonomous thermal dissipation techniques for the Titan-X9 silicon layout. Closing ou..."



---
### Key Takeaways for Students
1. **Chunking** divides massive text corpora into focused, searchable units.
2. **Overlap** protects against lost context across chunk seams.
3. **Euclidean (L2) Distance** finds the closest match geometrically ($d=0.0$ means identical; smaller distance = closer match).
4. **The Scalability Challenge:** Computing distance across all vectors is a **linear scan** ($\mathcal{O}(N)$). While fast for 27 chunks, it becomes a bottleneck on millions of chunks.
5. **Next Step:** In **Lesson 04**, we introduce **ChromaDB**, an in-memory vector database that uses an **HNSW graph index** for sub-linear ($\mathcal{O}(\log N)$) search, and combine it with BM25 for **Hybrid Retrieval**!
